## Running analysis on a dataset

In [1]:
# Import the library used to interact with Cirro
from cirro import DataPortal

# Create a connection to Cirro with your identity
portal = DataPortal()

### Option 1 - run analysis using the same set of parameters used previously

In [2]:
# New dataset with FASTQs
input_dataset = portal.get_dataset(
    project="Pipeline Development",
    dataset="Test dataset for variant calling"
)
print(f"Dataset '{input_dataset.name}' contains {len(input_dataset.list_files()):,} files")

Dataset 'Test dataset for variant calling' contains 2 files


In [3]:
# Get the process to run on the dataset
process = portal.get_process_by_name('Align Reads (nf-core/sarek)')
print(f"Using the '{process.name}' process (ID: {process.id})")

Using the 'Align Reads (nf-core/sarek)' process (ID: process-nf-core-sarek-align-3-2)


In [4]:
# Previous dataset created by the pipeline
previous_run = portal.get_dataset(
    project="Pipeline Development",
    dataset="Genomic variant calling - parameter validation"
)
print(f"Using parameters from {previous_run.name}")
print(previous_run.params)

Using parameters from Genomic variant calling - parameter validation
{'WORKFLOW_VERSION': '3.2.3', 'analysis_type': {'genome': 'GATK.GRCh38', 'wes': True, 'analysis_type': 'Germline Variant Calling', 'intervals': 's3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7/resources/data/references/genome_bed/GRCh38_Chr20/regions.bed', 'tools': ['strelka', 'haplotypecaller']}, 'annotation': {'annotation_tool': []}, 'read_trimming_options': {'trim_fastq': False}}


In [5]:
# Start a new run, using the parameters from the previous run
new_dataset_id = input_dataset.run_analysis(
    name="Genomic variant calling - new run",
    description='Test from SDK',
    process=process,
    params=previous_run.params
)
print(f"Started new analysis: ID {new_dataset_id}")

Started new analysis: ID f7ca7e1b-d64c-4747-b647-0e984db87aa5


### Option 2: Build parameters from scratch

In [6]:
# Get the project by name
project = portal.get_project_by_name('Pipeline Development') 
print(f"Project '{project.name}' contains {len(project.list_datasets()):,} datasets")

Project 'Pipeline Development' contains 709 datasets


In [7]:
# Get a particular dataset from that project
dataset = project.get_dataset_by_name('Test dataset for variant calling')
print(f"Dataset '{dataset.name}' contains {len(dataset.list_files()):,} files")

Dataset 'Test dataset for variant calling' contains 2 files


Look up the parameters that are required for the process. You'll have to set values for these parameters later.

In [8]:
param_spec = process.get_parameter_spec()
param_spec.print()

Parameters:
	Workflow Version (key=workflow_version, default=3.6.0, type=string, enum=['3.1', '3.1.1', '3.1.2', '3.2.3', '3.3.2', '3.4.4', '3.5.1', '3.6.0'], description=Select the specific version of nf-core/sarek used for analysis)
	Experimental Design (Group)
		Reference Genome (key=genome, default=GATK.GRCh38, type=string, enum=['GATK.GRCh38', 'GATK.GRCh37', 'GRCm38'])
		Whole Exome/Targeted Gene Panel Assay (key=wes, type=boolean, description=Please indicate if your data was generated using a capture kit.)
		Genomic intervals (key=intervals, type=string, description=Target bed file in case of whole exome or targeted sequencing or intervals file for parallelization.)
	Read Trimming Options (Group)
		Trim reads using Trim-Galore? (key=trim_fastq, type=boolean)
	Advanced Options (Group)
		MarkDuplicates - Optical Duplicate Pixel Distance (key=optical_duplicate_pixel_distance, default=100, type=integer, description=The `--OPTICAL_DUPLICATE_PIXEL_DISTANCE` parameter is used by MarkDupl

Look up the references you'll need to use as input parameters. See the [Using_references](Using_references.ipynb) notebook for more info on how to find references

In [9]:
references = project.list_references('Genome Regions (BED)')
print("The BED references available are:\n" + "\n - ".join(list(map(str, references))))
reference_library = project.get_reference_by_name('GRCh38_Chr20', 'Genome Regions (BED)')

print(f"\nThe reference library we are using is: {reference_library.name}\nThe absolute path to the file is: {reference_library.absolute_path}")

The BED references available are:
wgs_calling_regions.hg19.bed
 - hg38
 - epi2me-labs-wf-human-variation-ref
 - wgs_calling_regions.hg38.bed
 - GRCh38_Chr20
 - NimbleGen_SeqCap_EZ_Exome_primary-capture_hg19_chr17

The reference library we are using is: GRCh38_Chr20
The absolute path to the file is: s3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7/resources/data/references/genome_bed/GRCh38_Chr20/regions.bed


Define the parameters you want to use. The keys you'll want to use will come from the `param_spec` variable defined above (look at the `key` for each entry).

In [10]:
params = {
    'WORKFLOW_VERSION': '3.2.3',
    'analysis_type': {
        'genome': 'GATK.GRCh38',
        'wes': True,
        'analysis_type': 'Germline Variant Calling',
        'intervals': 's3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7/resources/data/references/genome_bed/GRCh38_Chr20/regions.bed',
        'tools': ['strelka', 'haplotypecaller']
    },
    'annotation': {
        'annotation_tool': []
    },
    'read_trimming_options': {
        'trim_fastq': False
    }
}
params

{'WORKFLOW_VERSION': '3.2.3',
 'analysis_type': {'genome': 'GATK.GRCh38',
  'wes': True,
  'analysis_type': 'Germline Variant Calling',
  'intervals': 's3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7/resources/data/references/genome_bed/GRCh38_Chr20/regions.bed',
  'tools': ['strelka', 'haplotypecaller']},
 'annotation': {'annotation_tool': []},
 'read_trimming_options': {'trim_fastq': False}}

Before submitting the analysis, the client automatically validates that the parameters are valid.
But, you can also validate them manually using `validate_params`

In [11]:
try:
    param_spec.validate_params({
        'library': 1
    })
except Exception as e:
    print(e)

Run the analysis using the process, dataset, project, and parameters you defined above.

In [12]:
# Run the analysis, specifying a name and description for the resulting dataset
new_dataset_id = input_dataset.run_analysis(
    name='Variant Calling Analysis',
    description='Test from SDK',
    process=process,
    params=params
)
print(new_dataset_id)

ca8eee87-09d9-4abe-ba0e-4e6ba48b33fa
